# Dataset review

This notebook starts the dataset review for the European Power Mix Analysis project.

This is an investigation on whether nuclear-heavy European electricity systems are associated with cheaper, cleaner or less volatile electricity, and what Austria’s non-nuclear model reveals.

Countries included:
- Austria
- United Kingdom
- France
- Germany
- Czechia
- Slovakia

In [9]:
import pandas as pd
import numpy as np

In [10]:
countries = [
    "Austria",
    "United Kingdom",
    "France",
    "Germany",
    "Czechia",
    "Slovakia"
]

## Data source plan

First dataset: Ember electricity data.

Aim:
- Load generation mix data.
- Filter to the selected countries.
- Compare nuclear, hydro, gas, fossil and renewable shares.
- Keep price and carbon interpretation separate until source checks are complete.

In [11]:
from pathlib import Path

project_root = Path.cwd().parent
project_root

WindowsPath('C:/Users/Tristan/Projects/nuclear-prices-carbon-europe')

In [12]:
data_raw = project_root / "data" / "raw"
data_processed = project_root / "data" / "processed"

data_raw, data_processed

(WindowsPath('C:/Users/Tristan/Projects/nuclear-prices-carbon-europe/data/raw'),
 WindowsPath('C:/Users/Tristan/Projects/nuclear-prices-carbon-europe/data/processed'))

## Load Ember data

This section loads the first raw electricity dataset and checks the columns, countries, units and available years before doing any analysis 

In [13]:
raw_files = list(data_raw.glob("*"))

raw_files

[WindowsPath('C:/Users/Tristan/Projects/nuclear-prices-carbon-europe/data/raw/ember'),
 WindowsPath('C:/Users/Tristan/Projects/nuclear-prices-carbon-europe/data/raw/eurostat'),
 WindowsPath('C:/Users/Tristan/Projects/nuclear-prices-carbon-europe/data/raw/iaea_pris')]

In [14]:
ember_dir = data_raw / "ember"

ember_files = [p for p in ember_dir.rglob("*") if p.is_file()]
ember_files

[WindowsPath('C:/Users/Tristan/Projects/nuclear-prices-carbon-europe/data/raw/ember/ember_yearly_electricity_data.csv')]

In [15]:
file_path = ember_files[0]

df = pd.read_csv(file_path)

df.head()

,Area,ISO 3 code,Year,Area type,Continent,Ember region,EU,OECD,G20,G7,ASEAN,Category,Subcategory,Variable,Unit,Value,YoY absolute change,YoY % change
0,Afghanistan,AFG,2000,Country or economy,Asia,Asia,0.0,0.0,0.0,0.0,0.0,Capacity,Aggregate fuel,Clean,GW,0.19,NaN,NaN
1,Afghanistan,AFG,2000,Country or economy,Asia,Asia,0.0,0.0,0.0,0.0,0.0,Capacity,Aggregate fuel,Fossil,GW,0.03,NaN,NaN
2,Afghanistan,AFG,2000,Country or economy,Asia,Asia,0.0,0.0,0.0,0.0,0.0,Capacity,Aggregate fuel,Gas and Other Fossil,GW,0.03,NaN,NaN
3,Afghanistan,AFG,2000,Country or economy,Asia,Asia,0.0,0.0,0.0,0.0,0.0,Capacity,Aggregate fuel,"Hydro, Bioenergy and Other Renewables",GW,0.19,NaN,NaN
4,Afghanistan,AFG,2000,Country or economy,Asia,Asia,0.0,0.0,0.0,0.0,0.0,Capacity,Aggregate fuel,Renewables,GW,0.19,NaN,NaN


In [16]:
df.shape

(371020, 18)

In [17]:
df.columns

Index(['Area', 'ISO 3 code', 'Year', 'Area type', 'Continent', 'Ember region',
       'EU', 'OECD', 'G20', 'G7', 'ASEAN', 'Category', 'Subcategory',
       'Variable', 'Unit', 'Value', 'YoY absolute change', 'YoY % change'],
      dtype='object')

In [18]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

df.columns

Index(['area', 'iso_3_code', 'year', 'area_type', 'continent', 'ember_region',
       'eu', 'oecd', 'g20', 'g7', 'asean', 'category', 'subcategory',
       'variable', 'unit', 'value', 'yoy_absolute_change', 'yoy_%_change'],
      dtype='object')

In [19]:
df_selected = df[df["area"].isin(countries)].copy()

df_selected.shape

(10167, 18)

In [20]:
df_selected["area"].unique()

array(['Austria', 'Czechia', 'France', 'Germany', 'Slovakia',
       'United Kingdom'], dtype=object)

In [21]:
df_selected["category"].value_counts()

category
Electricity generation    4816
Power sector emissions    2642
Capacity                  2241
Electricity demand         312
Electricity imports        156
Name: count, dtype: int64

In [22]:
df_selected["variable"].value_counts().head(40)

variable
Clean                                    618
Coal                                     618
Solar                                    618
Other Fossil                             618
Fossil                                   618
Hydro                                    618
Gas                                      618
Nuclear                                  618
Bioenergy                                618
Wind and Solar                           618
Renewables                               618
Hydro, Bioenergy and Other Renewables    618
Gas and Other Fossil                     618
Wind                                     602
Other Renewables                         595
Demand                                   156
Demand per capita                        156
Total Generation                         156
Net Imports                              156
CO2 intensity                            156
Total emissions                          156
Name: count, dtype: int64

In [23]:
df_selected["unit"].value_counts()

unit
TWh         2798
mtCO2       2486
%           2330
GW          2241
MWh          156
gCO2/kWh     156
Name: count, dtype: int64

## First generation mix table

This section filters Ember electricity generation data to the selected countries and key generation sources.

This is measured generation data, not a claim that any one technology causes lower prices or lower emissions.

In [25]:
df_selected["year"].min(), df_selected["year"].max()

(2000, 2025)

In [26]:
latest_year = df_selected["year"].max()
latest_year

2025

The latest year was selected. Electricity in TWh:

In [27]:
generation = df_selected[
    (df_selected["category"] == "Electricity generation") &
    (df_selected["unit"] == "TWh") &
    (df_selected["year"] == latest_year)
].copy()

In [28]:
generation.head()

,area,iso_3_code,year,area_type,continent,ember_region,eu,oecd,g20,g7,asean,category,subcategory,variable,unit,value,yoy_absolute_change,yoy_%_change
23109,Austria,AUT,2025,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Aggregate fuel,Clean,TWh,61.21,-6.45,-9.53
23110,Austria,AUT,2025,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Aggregate fuel,Fossil,TWh,12.01,1.20,11.10
23111,Austria,AUT,2025,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Aggregate fuel,Gas and Other Fossil,TWh,12.01,1.20,11.10
23112,Austria,AUT,2025,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Aggregate fuel,"Hydro, Bioenergy and Other Renewables",TWh,42.59,-7.79,-15.46
23113,Austria,AUT,2025,Country or economy,Europe,Europe,1.0,1.0,0.0,0.0,0.0,Electricity generation,Aggregate fuel,Renewables,TWh,61.21,-6.45,-9.53


Main sources of Electricity

In [30]:
main_sources = [
    "Nuclear",
    "Hydro",
    "Gas",
    "Coal",
    "Wind",
    "Solar",
    "Bioenergy",
    "Other Fossil",
    "Other Renewables"
]

In [31]:
generation_main = generation[generation["variable"].isin(main_sources)].copy()

In [32]:
generation_main[["area", "year", "variable", "unit", "value"]].head(20)

,area,year,variable,unit,value
23124,Austria,2025,Bioenergy,TWh,4.66
23125,Austria,2025,Coal,TWh,0.00
23126,Austria,2025,Gas,TWh,8.74
23127,Austria,2025,Hydro,TWh,37.93
23128,Austria,2025,Nuclear,TWh,0.00
23129,Austria,2025,Other Fossil,TWh,3.27
23130,Austria,2025,Other Renewables,TWh,0.00
23131,Austria,2025,Solar,TWh,10.31
23132,Austria,2025,Wind,TWh,8.31
87172,Czechia,2025,Bioenergy,TWh,6.05


In [33]:
generation_mix_table = generation_main.pivot_table(
    index="area",
    columns="variable",
    values="value",
    aggfunc="sum"
).fillna(0)

generation_mix_table

variable,Bioenergy,Coal,Gas,Hydro,Nuclear,Other Fossil,Other Renewables,Solar,Wind
area,,,,,,,,,
Austria,4.66,0.00,8.74,37.93,0.00,3.27,0.00,10.31,8.31
Czechia,6.05,26.60,4.08,1.71,31.94,0.16,0.00,4.39,0.57
France,10.27,1.74,17.25,59.41,392.07,10.36,0.57,31.82,46.49
Germany,50.53,103.15,82.67,19.56,0.00,18.91,0.00,89.62,136.03
Slovakia,1.58,0.31,3.28,3.16,19.33,0.73,0.01,0.70,0.00
United Kingdom,41.21,0.33,90.91,5.55,36.38,12.77,0.00,19.32,85.84


In [34]:
generation_mix_share = generation_mix_table.div(
    generation_mix_table.sum(axis=1),
    axis=0
) * 100

generation_mix_share.round(1)

variable,Bioenergy,Coal,Gas,Hydro,Nuclear,Other Fossil,Other Renewables,Solar,Wind
area,,,,,,,,,
Austria,6.4,0.0,11.9,51.8,0.0,4.5,0.0,14.1,11.3
Czechia,8.0,35.2,5.4,2.3,42.3,0.2,0.0,5.8,0.8
France,1.8,0.3,3.0,10.4,68.8,1.8,0.1,5.6,8.2
Germany,10.1,20.6,16.5,3.9,0.0,3.8,0.0,17.9,27.2
Slovakia,5.4,1.1,11.3,10.9,66.4,2.5,0.0,2.4,0.0
United Kingdom,14.1,0.1,31.1,1.9,12.4,4.4,0.0,6.6,29.4


## Initial observations*

For the latest available year (2025) in the Ember dataset:

- Austria appears hydro-heavy, with no nuclear generation in this table
- France and Slovakia are on the side of nuclear-heavy when compared to other countries
- Czechia combines nuclear with a large coal share
- Germany has high wind and solar shares, but still shows material coal and gas generation
- The United Kingdom shows a large gas and wind share, with a smaller nuclear share

*These are descriptive observations only. They do not show that any technology causes lower prices, lower emissions or lower volatility

In [35]:
key_mix_summary = generation_mix_share[
    ["Nuclear", "Hydro", "Gas", "Coal", "Wind", "Solar"]
].round(1)

key_mix_summary

variable,Nuclear,Hydro,Gas,Coal,Wind,Solar
area,,,,,,
Austria,0.0,51.8,11.9,0.0,11.3,14.1
Czechia,42.3,2.3,5.4,35.2,0.8,5.8
France,68.8,10.4,3.0,0.3,8.2,5.6
Germany,0.0,3.9,16.5,20.6,27.2,17.9
Slovakia,66.4,10.9,11.3,1.1,0.0,2.4
United Kingdom,12.4,1.9,31.1,0.1,29.4,6.6


In [41]:
key_mix_summary.sort_values("Nuclear", ascending=False)

variable,Nuclear,Hydro,Gas,Coal,Wind,Solar
area,,,,,,
France,68.8,10.4,3.0,0.3,8.2,5.6
Slovakia,66.4,10.9,11.3,1.1,0.0,2.4
Czechia,42.3,2.3,5.4,35.2,0.8,5.8
United Kingdom,12.4,1.9,31.1,0.1,29.4,6.6
Austria,0.0,51.8,11.9,0.0,11.3,14.1
Germany,0.0,3.9,16.5,20.6,27.2,17.9
